# AST Fase 2 — Gammatone con N_UNFREEZE=0 (solo cabeza lineal)

**Objetivo:** Comparar si congelar todo el backbone (solo entrenar la cabeza)
da mejores resultados que descongelar 3 capas (que generó overfitting severo).

**Filtro:** Solo Gammatone (ganador claro con N_UNFREEZE=3)  
**Duraciones:** 2s · 3s · 4s · 10s  
**N_UNFREEZE:** 0 — backbone 100% congelado, solo se entrena la cabeza lineal  
**Cabeza:** Linear(768→256)→ReLU→Dropout(0.5)→Linear(256→1)→Sigmoid  
**Validación:** 5-Fold × 2 reps = 10 folds  

Resultados se guardan en `Ansiedad/AST_fase2_unfreeze0/gammatone/`  
para no pisar los resultados anteriores.

---
Correr celdas en orden. Soporta interrupción y reanudación via checkpoint.

In [1]:
# ─── Celda 1: Imports y configuración ─────────────────────────────────────
import gc
import re
import json
import time
import csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import ASTConfig, ASTModel

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, auc
)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════════════
CONDITION    = "Ansiedad"
SPECS_ROOT   = Path("Clasificación Final/Ansiedad/specs")
RESULTS_ROOT = Path("Ansiedad/AST_fase2_unfreeze0")

PRETRAINED_NAME  = "MIT/ast-finetuned-audioset-10-10-0.4593"
OLD_FREQ_PATCHES = 12
OLD_TIME_PATCHES = 101

# Solo gammatone
FILTER_CONFIGS = [
    {"name": "gammatone", "img_h": 128, "ast_h": 128},
]

DURATIONS  = [10, 2, 3, 4]
N_SPLITS   = 5
N_REPEATS  = 2
BATCH_SIZE = 32
EPOCHS     = 30          # mas epocas porque la cabeza aprende mas lento
PATIENCE   = 6
VAL_FRAC   = 0.15
N_UNFREEZE = 0           # ← CAMBIO CLAVE: backbone 100% congelado
LR         = 1e-4        # lr mas alto porque solo entrenamos la cabeza
THR_GRID   = np.arange(0.05, 0.96, 0.02)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print(f"N_UNFREEZE={N_UNFREEZE} — backbone congelado, solo cabeza entrena")
print("Configuracion cargada")

/home/ci2dt2-ai/Proyectos/Psiquiatria/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo: cuda
  GPU: NVIDIA GeForce RTX 5080
N_UNFREEZE=0 — backbone congelado, solo cabeza entrena
Configuracion cargada


In [2]:
# ─── Celda 2: Dataset ──────────────────────────────────────────────────────

def build_file_index(root: Path):
    img_map    = {}
    stem_label = {}
    for cls_str in ["0", "1"]:
        d = root / cls_str
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        for p in sorted(d.glob("*.png")):
            stem = re.sub(r'_seg\d+$', '', p.stem)
            img_map.setdefault(stem, []).append(p)
            stem_label[stem] = int(cls_str)
    stems  = sorted(img_map.keys())
    labels = np.array([stem_label[s] for s in stems], dtype=np.int64)
    return stems, labels, img_map


def get_target_width(img_map: dict) -> int:
    max_w = 0
    for paths in img_map.values():
        for p in paths:
            w = Image.open(p).size[0]
            if w > max_w:
                max_w = w
            break
    return max_w


def compute_patch_counts(ast_h, target_w, patch_size=16, stride=10):
    return (ast_h - patch_size) // stride + 1, (target_w - patch_size) // stride + 1


BASE_TRANSFORM_FN = lambda: __import__('torchvision').transforms.Compose([
    __import__('torchvision').transforms.Grayscale(num_output_channels=1),
    __import__('torchvision').transforms.ToTensor(),
    __import__('torchvision').transforms.Normalize(mean=[0.5], std=[0.5]),
])

import torchvision.transforms as T
BASE_TRANSFORM = T.Compose([
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5]),
])


class ASTSpectrogramDataset(Dataset):
    def __init__(self, stems, img_map, ast_h, target_w):
        self.samples  = []
        self.ast_h    = ast_h
        self.target_w = target_w
        for stem in stems:
            label = int(img_map[stem][0].parent.name)
            for p in img_map[stem]:
                self.samples.append((p, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        tensor = BASE_TRANSFORM(Image.open(path)).squeeze(0)
        if tensor.shape[0] < self.ast_h:
            tensor = F.pad(tensor, (0, 0, 0, self.ast_h - tensor.shape[0]), value=-1.0)
        if tensor.shape[1] < self.target_w:
            tensor = F.pad(tensor, (0, self.target_w - tensor.shape[1]), value=-1.0)
        return tensor.T, torch.tensor(label, dtype=torch.long)


class ASTStemDataset(Dataset):
    def __init__(self, stems, img_map, ast_h, target_w):
        self.samples  = []
        self.ast_h    = ast_h
        self.target_w = target_w
        for stem in stems:
            for p in img_map[stem]:
                self.samples.append((p, stem))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, stem = self.samples[idx]
        tensor = BASE_TRANSFORM(Image.open(path)).squeeze(0)
        if tensor.shape[0] < self.ast_h:
            tensor = F.pad(tensor, (0, 0, 0, self.ast_h - tensor.shape[0]), value=-1.0)
        if tensor.shape[1] < self.target_w:
            tensor = F.pad(tensor, (0, self.target_w - tensor.shape[1]), value=-1.0)
        return tensor.T, stem

print("Dataset definido")

Dataset definido


In [3]:
# ─── Celda 3: Modelo ASTClassifier con N_UNFREEZE=0 ───────────────────────

class ASTClassifier(nn.Module):
    def __init__(self, ast_h, target_w, n_unfreeze=N_UNFREEZE):
        super().__init__()

        freq_patches = (ast_h    - 16) // 10 + 1
        time_patches = (target_w - 16) // 10 + 1
        print(f"  Patches: freq={freq_patches}  time={time_patches}  "
              f"total={freq_patches*time_patches}")

        config = ASTConfig(
            num_mel_bins     = ast_h,
            max_length       = target_w,
            hidden_size      = 768,
            num_hidden_layers = 12,
            num_attention_heads = 12,
            intermediate_size = 3072,
            patch_size       = 16,
            frequency_stride = 10,
            time_stride      = 10,
        )

        pretrained = ASTModel.from_pretrained(PRETRAINED_NAME)
        self.ast   = ASTModel(config)

        self.ast.encoder.load_state_dict(pretrained.encoder.state_dict())
        self.ast.layernorm.load_state_dict(pretrained.layernorm.state_dict())
        self.ast.embeddings.patch_embeddings.load_state_dict(
            pretrained.embeddings.patch_embeddings.state_dict())
        self.ast.embeddings.cls_token.data = \
            pretrained.embeddings.cls_token.data.clone()
        self.ast.embeddings.distillation_token.data = \
            pretrained.embeddings.distillation_token.data.clone()

        # Interpolar positional embeddings
        old_pos = pretrained.embeddings.position_embeddings.data
        new_pos = self._interp_pos_embed(
            old_pos, freq_patches, time_patches,
            OLD_FREQ_PATCHES, OLD_TIME_PATCHES)
        self.ast.embeddings.position_embeddings = nn.Parameter(new_pos)
        print(f"  Pos embed: {old_pos.shape} -> {new_pos.shape}")

        del pretrained
        gc.collect()

        # Cabeza con Dropout más alto para regularizar
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

        # Congelar TODO primero
        for param in self.parameters():
            param.requires_grad = False

        # Descongelar ultimas n_unfreeze capas del encoder (si n_unfreeze > 0)
        if n_unfreeze > 0:
            for layer in self.ast.encoder.layer[-n_unfreeze:]:
                for param in layer.parameters():
                    param.requires_grad = True
            for param in self.ast.layernorm.parameters():
                param.requires_grad = True

        # Descongelar SIEMPRE la cabeza
        for param in self.classifier.parameters():
            param.requires_grad = True

        n_total     = sum(p.numel() for p in self.parameters())
        n_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Total params     : {n_total:,}")
        print(f"  Entrenables      : {n_trainable:,} ({100*n_trainable/n_total:.2f}%)")

    @staticmethod
    def _interp_pos_embed(old_pos, new_f, new_t, old_f, old_t):
        special = old_pos[:, :2, :]
        patches = old_pos[:, 2:, :]
        hidden  = patches.shape[-1]
        patches = patches.reshape(1, old_f, old_t, hidden).permute(0, 3, 1, 2)
        patches = F.interpolate(patches.float(), size=(new_f, new_t),
                                mode='bilinear', align_corners=False)
        patches = patches.permute(0, 2, 3, 1).reshape(1, new_f * new_t, hidden)
        return torch.cat([special, patches], dim=1)

    def forward(self, x):
        outputs = self.ast(input_values=x)
        cls = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls).squeeze(1)

print("ASTClassifier definido (N_UNFREEZE=0)")

ASTClassifier definido (N_UNFREEZE=0)


In [4]:
# ─── Celda 4: Entrenamiento, inferencia, threshold ─────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds    = model(xb).view(-1)
        sample_w = cw_tensor[yb.long()]
        loss     = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        # Gradient clipping (buena práctica en transformers aunque backbone esté congelado)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def predict_stems(model, stems, img_map, ast_h, target_w):
    model.eval()
    ds = ASTStemDataset(stems, img_map, ast_h, target_w)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=4, pin_memory=True, persistent_workers=False)
    probs_by_stem = {s: [] for s in stems}
    with torch.no_grad():
        for xb, stem_names in dl:
            xb  = xb.to(DEVICE, non_blocking=True)
            out = model(xb).view(-1).cpu().numpy()
            for prob, stem in zip(out, stem_names):
                probs_by_stem[stem].append(float(prob))
    return {stem: float(np.mean(probs)) for stem, probs in probs_by_stem.items()}


def calibrate_threshold(model, val_stems, val_labels, img_map, ast_h, target_w):
    val_probs = predict_stems(model, val_stems, img_map, ast_h, target_w)
    probs_arr = np.array([val_probs[s] for s in val_stems])
    best_thr, best_f1 = 0.5, -1.0
    for thr in THR_GRID:
        preds = (probs_arr > thr).astype(int)
        score = f1_score(val_labels, preds, average='macro', zero_division=0)
        if score > best_f1:
            best_f1  = score
            best_thr = float(thr)
    return best_thr

print("Funciones de entrenamiento definidas")

Funciones de entrenamiento definidas


In [5]:
# ─── Celda 5: Checkpoint ───────────────────────────────────────────────────

def ckpt_path(out_dir): return out_dir / "_checkpoint.json"

def load_checkpoint(out_dir):
    p = ckpt_path(out_dir)
    if p.exists():
        with open(p) as f: data = json.load(f)
        print(f"  [CKPT] Reanudando desde fold {data['folds_done']}/{N_SPLITS*N_REPEATS}")
        return data
    return {"folds_done": 0, "fold_metrics": []}

def save_checkpoint(out_dir, ckpt):
    with open(ckpt_path(out_dir), "w") as f: json.dump(ckpt, f)

print("Checkpoint definido")

Checkpoint definido


In [6]:
# ─── Celda 6: Reporte y figuras ────────────────────────────────────────────

def save_results(out_dir, filter_name, dur, ckpt, n_splits, n_repeats):
    fold_metrics = ckpt["fold_metrics"]
    total_folds  = n_splits * n_repeats
    model_label  = f"AST-B unfreeze0 [{filter_name}] {dur}s"
    fname        = f"ast_uf0_{filter_name}_{dur}s"

    keys = ["acc", "prec", "rec", "f1", "f1mac", "auc"]
    metrics_arr = {k: np.array([d[k] for d in fold_metrics]) for k in keys}
    thresholds  = np.array([d["threshold"] for d in fold_metrics])

    print(f"\n{'='*72}")
    print(f"RESULTADOS — {model_label}")
    print(f"{n_splits}-Fold x {n_repeats} reps = {total_folds} folds")
    print(f"{'='*72}")
    print(f"\n  {'Rep-Fold':10s}  {'Thr':5s}  Acc    Prec   Rec    F1     F1-Mac AUC")
    for d in fold_metrics:
        print(f"  R{d['rep']:02d}-F{d['fold']:02d}     "
              f"{d['threshold']:.2f}   "
              f"{d['acc']:.3f}  {d['prec']:.3f}  {d['rec']:.3f}  "
              f"{d['f1']:.3f}  {d['f1mac']:.3f}  {d['auc']:.3f}")

    print(f"\n  {'─'*66}")
    print(f"  {'MEDIA':10s}  {thresholds.mean():.2f}   "
          + "  ".join(f"{metrics_arr[k].mean():.3f}" for k in keys))
    print(f"  {'± STD':10s}  {thresholds.std():.2f}   "
          + "  ".join(f"{metrics_arr[k].std():.3f}" for k in keys))

    # Comparacion directa con N_UNFREEZE=3
    prev = {"gammatone": {"2s": (0.570, 0.615), "3s": (0.494, 0.636),
                           "4s": (0.531, 0.590), "10s": (0.490, 0.544)}}
    dur_str = str(dur) + "s"
    if filter_name in prev and dur_str in prev[filter_name]:
        prev_f1, prev_auc = prev[filter_name][dur_str]
        curr_f1  = metrics_arr["f1mac"].mean()
        curr_auc = metrics_arr["auc"].mean()
        delta_f1  = curr_f1  - prev_f1
        delta_auc = curr_auc - prev_auc
        print(f"\n  Comparacion vs N_UNFREEZE=3:")
        print(f"  F1-Mac: {prev_f1:.3f} -> {curr_f1:.3f}  (delta={delta_f1:+.3f})"
              + (" MEJOR" if delta_f1 > 0.01 else " PEOR" if delta_f1 < -0.01 else " SIMILAR"))
        print(f"  AUC   : {prev_auc:.3f} -> {curr_auc:.3f}  (delta={delta_auc:+.3f})"
              + (" MEJOR" if delta_auc > 0.01 else " PEOR" if delta_auc < -0.01 else " SIMILAR"))

    # Figuras
    labels_m = {"acc": "Accuracy", "prec": "Precision", "rec": "Recall",
                 "f1": "F1 (pos)", "f1mac": "F1-Macro", "auc": "AUC-ROC"}
    fig, axes = plt.subplots(1, 6, figsize=(20, 4))
    fig.suptitle(f"{model_label} — {total_folds} folds", fontsize=11)
    for ax, (k, lbl) in zip(axes, labels_m.items()):
        ax.boxplot(metrics_arr[k], patch_artist=True,
                   boxprops=dict(facecolor='lightgreen'),
                   medianprops=dict(color='darkgreen', linewidth=2))
        ax.set_title(lbl, fontsize=10); ax.set_ylim(0, 1.05); ax.set_xticks([])
        ax.axhline(metrics_arr[k].mean(), color='red', linestyle='--',
                   linewidth=1, label=f"u={metrics_arr[k].mean():.3f}")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(out_dir / f"boxplot_{fname}.png", dpi=150); plt.show(); plt.close()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(thresholds, bins=10, color='seagreen', edgecolor='white', alpha=0.8)
    ax.axvline(thresholds.mean(), color='darkred', linestyle='--',
               label=f"u={thresholds.mean():.2f}  sd={thresholds.std():.2f}")
    ax.set_xlabel("Threshold optimo por fold"); ax.set_ylabel("Frecuencia")
    ax.set_title(f"Thresholds — {model_label}"); ax.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"thresholds_{fname}.png", dpi=150); plt.show(); plt.close()

    cms     = np.array([d["cm"] for d in fold_metrics], dtype=float)
    cm_mean = cms.mean(axis=0); cm_std = cms.std(axis=0)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_mean, annot=False, cmap='Greens', ax=ax,
                xticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"],
                yticklabels=[f"Sin {CONDITION}", f"Con {CONDITION}"])
    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.5, f"{cm_mean[i,j]:.1f}\n+/-{cm_std[i,j]:.1f}",
                    ha='center', va='center', fontsize=12,
                    color='white' if cm_mean[i,j] > cm_mean.max()/2 else 'black')
    ax.set_title(f"Confusion media — {model_label}")
    ax.set_ylabel("Real"); ax.set_xlabel("Prediccion")
    plt.tight_layout()
    plt.savefig(out_dir / f"confusion_{fname}.png", dpi=150); plt.show(); plt.close()

    fig, ax = plt.subplots(figsize=(8, 7))
    for d in fold_metrics:
        ax.plot(np.array(d["roc_fpr"]), np.array(d["roc_tpr"]),
                alpha=0.3, lw=1, color='seagreen')
    mean_fpr = np.linspace(0, 1, 100)
    tprs_interp = [np.interp(mean_fpr, d["roc_fpr"], d["roc_tpr"]) for d in fold_metrics]
    mean_tpr = np.mean(tprs_interp, axis=0)
    std_tpr  = np.std(tprs_interp, axis=0)
    aucs = [d["auc"] for d in fold_metrics]
    ax.plot(mean_fpr, mean_tpr, color='darkgreen', lw=2,
            label=f"Media AUC = {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}")
    ax.fill_between(mean_fpr, mean_tpr-std_tpr, mean_tpr+std_tpr,
                    alpha=0.15, color='darkgreen', label="+/- 1 std")
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_xlabel("Falsos Positivos"); ax.set_ylabel("Verdaderos Positivos")
    ax.set_title(f"ROC — {model_label}"); ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(out_dir / f"roc_{fname}.png", dpi=150); plt.show(); plt.close()

    csv_path = out_dir / f"metrics_{fname}.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["rep","fold","threshold",
                                                "acc","prec","rec","f1","f1mac","auc"])
        writer.writeheader()
        writer.writerows([{k: d[k] for k in ["rep","fold","threshold",
                                               "acc","prec","rec","f1","f1mac","auc"]}
                           for d in fold_metrics])
    print(f"  Guardado en : {out_dir}")

print("save_results definida")

save_results definida


In [7]:
# ─── Celda 7: LOOP PRINCIPAL ───────────────────────────────────────────────

TOTAL_CONFIGS = len(FILTER_CONFIGS) * len(DURATIONS)
config_n = 0

for filt_cfg in FILTER_CONFIGS:
    filter_name = filt_cfg["name"]
    ast_h       = filt_cfg["ast_h"]

    for dur in DURATIONS:
        config_n   += 1
        image_root  = SPECS_ROOT / filter_name / f"{dur}s"
        out_dir     = RESULTS_ROOT / filter_name / f"{dur}s"
        out_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n{'='*72}")
        print(f"  CONFIG {config_n}/{TOTAL_CONFIGS} — "
              f"filtro={filter_name}  dur={dur}s  N_UNFREEZE={N_UNFREEZE}")
        print(f"{'='*72}")

        if not image_root.exists():
            print(f"  [SKIP] No existe: {image_root}"); continue

        file_stems, file_labels, img_map = build_file_index(image_root)
        if len(file_stems) == 0:
            print(f"  [SKIP] Sin imagenes"); continue

        target_w   = get_target_width(img_map)
        f_p, t_p   = compute_patch_counts(ast_h, target_w)
        total_imgs = sum(len(v) for v in img_map.values())
        print(f"\n  Pacientes : {len(file_stems)}  "
              f"(sin={int((file_labels==0).sum())} / con={int((file_labels==1).sum())})")
        print(f"  Segmentos : {total_imgs:,}  (~{total_imgs/len(file_stems):.1f}/audio)")
        print(f"  AST input : ({target_w}, {ast_h})  patches: {f_p}x{t_p}={f_p*t_p}")

        ckpt       = load_checkpoint(out_dir)
        folds_done = ckpt["folds_done"]
        total_folds = N_SPLITS * N_REPEATS

        if folds_done >= total_folds:
            print(f"  [SKIP] Ya completado. Regenerando figuras...")
            save_results(out_dir, filter_name, dur, ckpt, N_SPLITS, N_REPEATS)
            continue

        rskf = RepeatedStratifiedKFold(
            n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=42)
        file_stems_arr = np.array(file_stems)

        print(f"\n  CV: {N_SPLITS}-Fold x {N_REPEATS} reps = {total_folds} folds")
        if folds_done > 0:
            print(f"  Reanudando desde fold {folds_done+1}/{total_folds}")

        for fold_idx, (tr_idx, te_idx) in enumerate(
                rskf.split(file_stems_arr, file_labels)):

            if fold_idx < folds_done:
                continue

            repeat_n    = fold_idx // N_SPLITS + 1
            fold_n      = fold_idx %  N_SPLITS + 1
            fold_global = fold_idx + 1

            print(f"\n  Rep {repeat_n}/{N_REPEATS}  |  "
                  f"Fold {fold_n}/{N_SPLITS}  [{fold_global}/{total_folds}]")

            tr_stems  = list(file_stems_arr[tr_idx])
            te_stems  = list(file_stems_arr[te_idx])
            tr_labels = file_labels[tr_idx]
            te_labels = file_labels[te_idx]

            sss = StratifiedShuffleSplit(
                n_splits=1, test_size=VAL_FRAC, random_state=42+fold_idx)
            tr_sub_idx, val_idx = next(sss.split(tr_stems, tr_labels))

            val_stems       = [tr_stems[i] for i in val_idx]
            val_labels_fold = tr_labels[val_idx]
            train_stems     = [tr_stems[i] for i in tr_sub_idx]
            train_labels    = tr_labels[tr_sub_idx]

            train_ds = ASTSpectrogramDataset(train_stems, img_map, ast_h, target_w)
            train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=4, pin_memory=True,
                                  persistent_workers=False)
            print(f"  Train: {len(train_stems)} pac / {len(train_ds):,} segs  "
                  f"| Val: {len(val_stems)} pac  | Test: {len(te_stems)} pac")

            weights   = class_weight.compute_class_weight(
                            'balanced', classes=np.unique(train_labels), y=train_labels)
            cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

            model     = ASTClassifier(ast_h, target_w).to(DEVICE)
            # Adam (no AdamW) con LR mas alto porque solo es la cabeza
            optimizer = optim.Adam(model.parameters(), lr=LR)
            criterion = nn.BCELoss(reduction='none')
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                            optimizer, 'min', factor=0.5, patience=3, min_lr=1e-7)

            best_loss, patience_cnt, best_state = float('inf'), 0, None
            t0 = time.time()

            for epoch in range(EPOCHS):
                loss = train_one_epoch(
                    model, train_dl, optimizer, criterion, cw_tensor)
                scheduler.step(loss)
                if loss < best_loss:
                    best_loss    = loss
                    patience_cnt = 0
                    best_state   = {k: v.cpu().clone()
                                    for k, v in model.state_dict().items()}
                    mark = "v"
                else:
                    patience_cnt += 1
                    mark = f"({patience_cnt}/{PATIENCE})"
                print(f"  Ep {epoch+1:02d}/{EPOCHS}  "
                      f"loss {loss:.4f}  best {best_loss:.4f}  {mark}")
                if patience_cnt >= PATIENCE:
                    print("  Early stopping.")
                    break

            model.load_state_dict(best_state)
            print(f"  Entrenamiento: {time.time()-t0:.1f}s")

            best_thr = calibrate_threshold(
                model, val_stems, val_labels_fold, img_map, ast_h, target_w)
            print(f"  Threshold optimo: {best_thr:.2f}")

            stem_probs = predict_stems(model, te_stems, img_map, ast_h, target_w)
            fold_true, fold_pred, fold_prob = [], [], []
            for stem, lbl in zip(te_stems, te_labels):
                prob = stem_probs[stem]
                fold_true.append(int(lbl))
                fold_pred.append(1 if prob > best_thr else 0)
                fold_prob.append(prob)

            fold_true_arr = np.array(fold_true)
            fold_prob_arr = np.array(fold_prob)

            if len(np.unique(fold_true_arr)) > 1:
                fpr, tpr, _ = roc_curve(fold_true_arr, fold_prob_arr)
                fold_auc    = float(auc(fpr, tpr))
                fpr_list, tpr_list = fpr.tolist(), tpr.tolist()
            else:
                fold_auc    = 0.0
                fpr_list, tpr_list = [0.0, 1.0], [0.0, 1.0]
                print("  [AVISO] Test de este fold tiene una sola clase — AUC=0")

            cm_fold = confusion_matrix(fold_true, fold_pred, labels=[0,1]).tolist()

            m = {
                "rep"      : repeat_n,
                "fold"     : fold_n,
                "threshold": round(best_thr, 4),
                "acc"      : float(accuracy_score(fold_true, fold_pred)),
                "prec"     : float(precision_score(fold_true, fold_pred, zero_division=0)),
                "rec"      : float(recall_score(fold_true, fold_pred, zero_division=0)),
                "f1"       : float(f1_score(fold_true, fold_pred, zero_division=0)),
                "f1mac"    : float(f1_score(fold_true, fold_pred,
                                            average='macro', zero_division=0)),
                "auc"      : fold_auc,
                "cm"       : cm_fold,
                "roc_fpr"  : fpr_list,
                "roc_tpr"  : tpr_list,
            }
            print(f"  acc={m['acc']:.3f}  prec={m['prec']:.3f}  "
                  f"rec={m['rec']:.3f}  F1={m['f1']:.3f}  "
                  f"F1-mac={m['f1mac']:.3f}  AUC={m['auc']:.3f}  "
                  f"thr={m['threshold']:.2f}")

            ckpt["fold_metrics"].append(m)
            ckpt["folds_done"] = fold_global
            save_checkpoint(out_dir, ckpt)

            del model, train_ds, train_dl, best_state, cw_tensor
            gc.collect()
            torch.cuda.empty_cache()

        save_results(out_dir, filter_name, dur, ckpt, N_SPLITS, N_REPEATS)

print(f"\n{'='*72}")
print(f"  COMPLETADO — resultados en: {RESULTS_ROOT}")
print(f"  Comparacion directa impresa al final de cada configuracion")
print(f"{'='*72}")


  CONFIG 1/4 — filtro=gammatone  dur=10s  N_UNFREEZE=0

  Pacientes : 79  (sin=60 / con=19)
  Segmentos : 1,259  (~15.9/audio)
  AST input : (313, 128)  patches: 12x30=360

  CV: 5-Fold x 2 reps = 10 folds

  Rep 1/2  |  Fold 1/5  [1/10]
  Train: 53 pac / 877 segs  | Val: 10 pac  | Test: 16 pac
  Patches: freq=12  time=30  total=360


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14297.38it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Pos embed: torch.Size([1, 1214, 768]) -> torch.Size([1, 362, 768])
  Total params     : 85,730,049
  Entrenables      : 197,121 (0.23%)
  Ep 01/30  loss 0.6780  best 0.6780  v
  Ep 02/30  loss 0.6354  best 0.6354  v
  Ep 03/30  loss 0.6156  best 0.6156  v
  Ep 04/30  loss 0.6097  best 0.6097  v
  Ep 05/30  loss 0.6105  best 0.6097  (1/6)
  Ep 06/30  loss 0.5818  best 0.5818  v
  Ep 07/30  loss 0.5724  best 0.5724  v
  Ep 08/30  loss 0.5600  best 0.5600  v
  Ep 09/30  loss 0.5577  best 0.5577  v
  Ep 10/30  loss 0.5564  best 0.5564  v
  Ep 11/30  loss 0.5412  best 0.5412  v
  Ep 12/30  loss 0.5257  best 0.5257  v
  Ep 13/30  loss 0.5152  best 0.5152  v
  Ep 14/30  loss 0.5179  best 0.5152  (1/6)
  Ep 15/30  loss 0.5101  best 0.5101  v
  Ep 16/30  loss 0.4993  best 0.4993  v
  Ep 17/30  loss 0.4823  best 0.4823  v
  Ep 18/30  loss 0.4954  best 0.4823  (1/6)
  Ep 19/30  loss 0.4775  best 0.4775  v
  Ep 20/30  loss 0.4689  best 0.4689  v
  Ep 21/30  loss 0.4853  best 0.4689  (1/6)
  Ep 2

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14084.11it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Pos embed: torch.Size([1, 1214, 768]) -> torch.Size([1, 362, 768])
  Total params     : 85,730,049
  Entrenables      : 197,121 (0.23%)
  Ep 01/30  loss 0.7022  best 0.7022  v
  Ep 02/30  loss 0.6969  best 0.6969  v
  Ep 03/30  loss 0.6705  best 0.6705  v
  Ep 04/30  loss 0.6631  best 0.6631  v
  Ep 05/30  loss 0.6457  best 0.6457  v
  Ep 06/30  loss 0.6437  best 0.6437  v
  Ep 07/30  loss 0.6223  best 0.6223  v
  Ep 08/30  loss 0.6137  best 0.6137  v
  Ep 09/30  loss 0.5992  best 0.5992  v
  Ep 10/30  loss 0.5919  best 0.5919  v
  Ep 11/30  loss 0.5809  best 0.5809  v
  Ep 12/30  loss 0.5786  best 0.5786  v
  Ep 13/30  loss 0.5538  best 0.5538  v
  Ep 14/30  loss 0.5735  best 0.5538  (1/6)
  Ep 15/30  loss 0.5603  best 0.5538  (2/6)
  Ep 16/30  loss 0.5493  best 0.5493  v
  Ep 17/30  loss 0.5514  best 0.5493  (1/6)
  Ep 18/30  loss 0.5342  best 0.5342  v
  Ep 19/30  loss 0.5451  best 0.5342  (1/6)
  Ep 20/30  loss 0.5142  best 0.5142  v
  Ep 21/30  loss 0.5080  best 0.5080  v
  Ep 2

...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19909.04it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Pos embed: torch.Size([1, 1214, 768]) -> torch.Size([1, 146, 768])
  Total params     : 85,564,161
  Entrenables      : 197,121 (0.23%)
  Ep 01/30  loss 0.6950  best 0.6950  v
  Ep 02/30  loss 0.6700  best 0.6700  v
  Ep 03/30  loss 0.6535  best 0.6535  v
  Ep 04/30  loss 0.6350  best 0.6350  v
  Ep 05/30  loss 0.6309  best 0.6309  v
  Ep 06/30  loss 0.6167  best 0.6167  v
  Ep 07/30  loss 0.6037  best 0.6037  v
  Ep 08/30  loss 0.5999  best 0.5999  v
  Ep 09/30  loss 0.5901  best 0.5901  v
  Ep 10/30  loss 0.5831  best 0.5831  v
  Ep 11/30  loss 0.5659  best 0.5659  v
  Ep 12/30  loss 0.5615  best 0.5615  v
  Ep 13/30  loss 0.5524  best 0.5524  v
  Ep 14/30  loss 0.5435  best 0.5435  v
  Ep 15/30  loss 0.5284  best 0.5284  v
  Ep 16/30  loss 0.5258  best 0.5258  v
  Ep 17/30  loss 0.5220  best 0.5220  v
  Ep 18/30  loss 0.5074  best 0.5074  v
  Ep 19/30  loss 0.4979  best 0.4979  v
  Ep 20/30  loss 0.4875  best 0.4875  v
  Ep 21/30  loss 0.4884  best 0.4875  (1/6)
  Ep 22/30  loss 0

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17837.42it/s]
[transformers] ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Pos embed: torch.Size([1, 1214, 768]) -> torch.Size([1, 146, 768])
  Total params     : 85,564,161
  Entrenables      : 197,121 (0.23%)
  Ep 01/30  loss 0.6829  best 0.6829  v
  Ep 02/30  loss 0.6430  best 0.6430  v
  Ep 03/30  loss 0.6319  best 0.6319  v
  Ep 04/30  loss 0.6143  best 0.6143  v
  Ep 05/30  loss 0.6090  best 0.6090  v
  Ep 06/30  loss 0.5919  best 0.5919  v
  Ep 07/30  loss 0.5862  best 0.5862  v
  Ep 08/30  loss 0.5825  best 0.5825  v
  Ep 09/30  loss 0.5680  best 0.5680  v
  Ep 10/30  loss 0.5592  best 0.5592  v
  Ep 11/30  loss 0.5609  best 0.5592  (1/6)
  Ep 12/30  loss 0.5493  best 0.5493  v
  Ep 13/30  loss 0.5407  best 0.5407  v
  Ep 14/30  loss 0.5350  best 0.5350  v
  Ep 15/30  loss 0.5367  best 0.5350  (1/6)
  Ep 16/30  loss 0.5272  best 0.5272  v
  Ep 17/30  loss 0.5220  best 0.5220  v
  Ep 18/30  loss 0.5212  best 0.5212  v
  Ep 19/30  loss 0.5066  best 0.5066  v
  Ep 20/30  loss 0.5003  best 0.5003  v
  Ep 21/30  loss 0.4938  best 0.4938  v
  Ep 22/30  lo

/tmp/ipykernel_931968/1105546159.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(out_dir / f"boxplot_{fname}.png", dpi=150); plt.show(); plt.close()
/tmp/ipykernel_931968/1105546159.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(out_dir / f"thresholds_{fname}.png", dpi=150); plt.show(); plt.close()
/tmp/ipykernel_931968/1105546159.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(out_dir / f"confusion_{fname}.png", dpi=150); plt.show(); plt.close()
/tmp/ipykernel_931968/1105546159.py:104: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(out_dir / f"roc_{fname}.png", dpi=150); plt.show(); plt.close()
